# Gráficos del modelo para la presentación

Lee de BigQuery las salidas de `sql/ml/` y genera los gráficos en `notebooks/figuras/`.

**Requisitos:** haber corrido `sql/ml/00_setup.sql`, `01_experiments.sql` y `02_train_final.sql`, y tener credenciales (`gcloud auth application-default login`).

```bash
.venv/bin/pip install -r notebooks/requirements.txt
GCP_PROJECT=<id-del-proyecto> .venv/bin/jupyter nbconvert --to notebook --execute --inplace notebooks/backtest_charts.ipynb
```

Con `MODO_PRUEBA=1` usa datos inventados para revisar el diseño de los gráficos sin conectarse a BigQuery; en ese modo no guarda figuras.

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = os.environ.get("GCP_PROJECT")
DATASET = "supply_chain"
INSUMO = "ELECTRODO_6013"  # insumo estrella del dashboard
MODO_PRUEBA = os.environ.get("MODO_PRUEBA") == "1"
FIG_DIR = Path("figuras")
if not MODO_PRUEBA:
    FIG_DIR.mkdir(exist_ok=True)

# Paleta (referencia dataviz, modo claro para diapositivas).
SUPERFICIE = "#fcfcfb"
TEXTO = "#0b0b0b"
TEXTO_2 = "#52514e"
GRILLA = "#e4e3df"
PRONOSTICO = "#2a78d6"   # azul: todo lo que sale del modelo
BANDA = "#cde2fb"        # azul claro: intervalo de predicción
REAL = "#52514e"         # gris tinta: lo observado

plt.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
    "axes.edgecolor": GRILLA, "axes.labelcolor": TEXTO_2, "axes.titlecolor": TEXTO,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.8,
    "xtick.color": TEXTO_2, "ytick.color": TEXTO_2, "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110, "savefig.dpi": 200,
})

MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

def eje_fechas(ax, con_anio=False, cada_dias=None):
    """Fechas del eje x en español: "15 oct" o "oct 2025"."""
    if cada_dias:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=cada_dias))
    def fmt(x, _):
        f = mdates.num2date(x)
        return f"{MESES[f.month - 1]} {f.year}" if con_anio else f"{f.day} {MESES[f.month - 1]}"
    ax.xaxis.set_major_formatter(plt.FuncFormatter(fmt))

def guardar(fig, nombre):
    if not MODO_PRUEBA:
        fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight")

## Carga de datos

In [ ]:
SQL = {
    "global": f"SELECT * FROM `{DATASET}.model_metrics`",
    "por_insumo": f"SELECT * FROM `{DATASET}.model_metrics_by_item`",
    "experimentos": f"""
        SELECT experimento, AVG(mape_semanal) AS mape_semanal, MAX(mape_semanal) AS mape_peor
        FROM `{DATASET}.ml_experiments` WHERE experimento != 'final'
        GROUP BY experimento""",
    "backtest": f"""
        SELECT id_insumo, fecha, real, pronostico
        FROM `{DATASET}.ml_backtest_points` WHERE experimento = 'final'
        ORDER BY id_insumo, fecha""",
    "explicacion": f"""
        SELECT DATE(time_series_timestamp) AS fecha, time_series_type AS tipo,
               time_series_data AS valor, trend AS tendencia,
               seasonal_period_weekly AS semanal, seasonal_period_yearly AS anual,
               holiday_effect AS feriados,
               prediction_interval_lower_bound AS lower_bound,
               prediction_interval_upper_bound AS upper_bound
        FROM ML.EXPLAIN_FORECAST(MODEL `{DATASET}.inventory_forecast_model`,
                                 STRUCT(30 AS horizon, 0.9 AS confidence_level))
        WHERE id_insumo = '{INSUMO}'
        ORDER BY fecha""",
    "arima": f"""
        SELECT id_insumo, non_seasonal_p AS p, non_seasonal_d AS d, non_seasonal_q AS q,
               has_drift, seasonal_periods, has_holiday_effect,
               has_spikes_and_dips, has_step_changes
        FROM ML.ARIMA_EVALUATE(MODEL `{DATASET}.inventory_forecast_model`,
                               STRUCT(FALSE AS show_all_candidate_models))
        ORDER BY id_insumo""",
}

def datos_prueba():
    """Datos inventados con la forma de las tablas reales (solo para revisar el diseño)."""
    rng = np.random.default_rng(0)
    insumos = ["ACERO_A36", "ACERO_INOX_304", "ALAMBRE_MIG_ER70S6", "DISCO_CORTE_7",
               "DISCO_DESBASTE_7", "ELECTRODO_6013", "ELECTRODO_7018", "GAS_ARGON",
               "GAS_CO2", "GUANTES_SOLDADOR", "PINTURA_ANTICORROSIVA", "TORNILLO_M10"]
    semana = np.array([1.0, 1.05, 1.05, 1.0, 0.9, 0.45, 0.05])
    test = pd.date_range("2026-10-04", "2026-10-31")
    filas = []
    for i, ins in enumerate(insumos):
        base = 20 + 10 * i
        esperado = base * semana[test.day_of_week]
        real = esperado * rng.gamma(10, 0.1, len(test))
        filas += [{"id_insumo": ins, "fecha": f, "real": r, "pronostico": e}
                  for f, r, e in zip(test, real, esperado)]
    backtest = pd.DataFrame(filas)
    mape = rng.uniform(4, 18, len(insumos))
    por_insumo = pd.DataFrame({"id_insumo": insumos, "mape": mape.round(1),
                               "precision": (100 - mape).round(1)})
    hist = pd.date_range("2023-11-01", "2026-10-31")
    fut = pd.date_range("2026-11-01", periods=30)
    fechas = hist.append(fut)
    t = np.arange(len(fechas))
    tendencia = 40 + 0.006 * t
    semanal = (semana[fechas.day_of_week] - semana.mean()) * 40
    anual = 6 * np.cos(2 * np.pi * (fechas.day_of_year - 270) / 365.25)
    feriados = np.where((fechas.month == 12) & (fechas.day >= 22), -25.0, 0.0)
    valor = np.maximum(0, tendencia + semanal + anual + feriados + rng.normal(0, 4, len(t)))
    es_fut = np.arange(len(fechas)) >= len(hist)
    explicacion = pd.DataFrame({
        "fecha": fechas, "tipo": np.where(es_fut, "forecast", "history"),
        "valor": valor, "tendencia": tendencia, "semanal": semanal, "anual": anual,
        "feriados": feriados,
        "lower_bound": np.where(es_fut, valor - 8 - 0.2 * (t - len(hist)), np.nan),
        "upper_bound": np.where(es_fut, valor + 8 + 0.2 * (t - len(hist)), np.nan)})
    return {
        "global": pd.DataFrame([{"modelo": "ARIMA_PLUS", "mape": mape.mean().round(1),
                                 "precision": (100 - mape.mean()).round(1),
                                 "evaluado_en": "2026-10-31"}]),
        "por_insumo": por_insumo,
        "experimentos": pd.DataFrame({"experimento": ["sin_limpieza", "por_defecto", "feriados_co"],
                                      "mape_semanal": [14.2, 10.8, 9.6], "mape_peor": [31.0, 22.5, 18.4]}),
        "backtest": backtest,
        "explicacion": explicacion,
        "arima": pd.DataFrame({"id_insumo": insumos}),
    }

if MODO_PRUEBA:
    d = datos_prueba()
else:
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    d = {k: client.query(q).to_dataframe() for k, q in SQL.items()}

for df in (d["backtest"], d["explicacion"]):
    df["fecha"] = pd.to_datetime(df["fecha"])
glob = d["global"].iloc[0]
print(f"Precisión global: {glob['precision']}% (MAPE semanal {glob['mape']}%)")

## 1. Precisión por insumo

In [ ]:
df = d["por_insumo"].sort_values("precision")
fig, ax = plt.subplots(figsize=(8, 5.2))
barras = ax.barh(df["id_insumo"], df["precision"], color=PRONOSTICO, height=0.6)
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in df["precision"]], padding=4, color=TEXTO, fontsize=9)
ax.set_xlim(0, 105)
ax.set_xlabel("Precisión (100 − MAPE semanal), %")
ax.grid(axis="y", visible=False)
ax.set_title(f"Precisión del modelo por insumo · promedio {glob['precision']:.1f}%")
fig.text(0.01, -0.02, "Backtest: el modelo no vio las últimas 4 semanas; se comparan totales semanales.",
         color=TEXTO_2, fontsize=9)
guardar(fig, "01_precision_por_insumo")
plt.show()

## 2. Comparación de configuraciones

In [ ]:
df = d["experimentos"].sort_values("mape_semanal", ascending=False)
fig, ax = plt.subplots(figsize=(8, 2.8))
barras = ax.barh(df["experimento"], df["mape_semanal"], color=PRONOSTICO, height=0.55)
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in df["mape_semanal"]], padding=4, color=TEXTO, fontsize=9)
ax.set_xlabel("MAPE semanal promedio, % (menor es mejor)")
ax.grid(axis="y", visible=False)
ax.set_title("Qué configuración de ARIMA_PLUS predice mejor")
guardar(fig, "02_experimentos")
plt.show()

## 3. Real vs. pronóstico en el backtest

In [ ]:
bt = d["backtest"]
uno = bt[bt["id_insumo"] == INSUMO]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(uno["fecha"], uno["real"], color=REAL, marker="o", markersize=4, label="Real")
ax.plot(uno["fecha"], uno["pronostico"], color=PRONOSTICO, linestyle="--", label="Pronóstico")
ultimo = uno.iloc[-1]
ax.annotate("Real", (ultimo["fecha"], ultimo["real"]), xytext=(6, 0), textcoords="offset points",
            color=TEXTO, va="center")
ax.annotate("Pronóstico", (ultimo["fecha"], ultimo["pronostico"]), xytext=(6, -12),
            textcoords="offset points", color=TEXTO, va="center")
ax.legend(loc="upper left", ncols=2)
eje_fechas(ax)
ax.set_ylabel("Consumo diario")
ax.set_ylim(bottom=0)
mape = d["por_insumo"].set_index("id_insumo").loc[INSUMO, "mape"]
ax.set_title(f"{INSUMO}: el modelo sigue el patrón semanal · MAPE semanal {mape:.1f}%")
guardar(fig, "03_backtest_insumo")
plt.show()

In [ ]:
insumos = sorted(bt["id_insumo"].unique())
mapes = d["por_insumo"].set_index("id_insumo")["mape"]
cols = 4
filas = int(np.ceil(len(insumos) / cols))
fig, axes = plt.subplots(filas, cols, figsize=(14, 2.6 * filas), sharex=True)
for ax, ins in zip(axes.flat, insumos):
    s = bt[bt["id_insumo"] == ins]
    ax.plot(s["fecha"], s["real"], color=REAL, linewidth=1.5, label="Real")
    ax.plot(s["fecha"], s["pronostico"], color=PRONOSTICO, linewidth=1.5, linestyle="--", label="Pronóstico")
    ax.set_title(f"{ins} · {mapes.get(ins, float('nan')):.1f}%", fontsize=10)
    ax.set_ylim(bottom=0)
    eje_fechas(ax, cada_dias=9)
    ax.tick_params(labelsize=8)
for ax in list(axes.flat)[len(insumos):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncols=2)
fig.suptitle("Backtest de los 12 insumos (título: MAPE semanal)", x=0.01, ha="left",
             fontsize=13, fontweight="bold", color=TEXTO)
fig.tight_layout(rect=(0, 0, 1, 0.96))
guardar(fig, "04_backtest_todos")
plt.show()

## 4. Pronóstico a 30 días con intervalo

In [ ]:
ex = d["explicacion"]
hist = ex[ex["tipo"] == "history"].tail(90)
fut = ex[ex["tipo"] == "forecast"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(fut["fecha"], fut["lower_bound"].clip(lower=0), fut["upper_bound"],
                color=BANDA, linewidth=0, label="Intervalo 90 %")
ax.plot(hist["fecha"], hist["valor"], color=REAL, label="Histórico")
ax.plot(fut["fecha"], fut["valor"].clip(lower=0), color=PRONOSTICO, linestyle="--", label="Pronóstico")
ax.axvline(fut["fecha"].iloc[0], color=GRILLA, linewidth=1)
ax.legend(loc="upper left", ncols=3)
eje_fechas(ax)
ax.set_ylabel("Consumo diario")
ax.set_ylim(bottom=0)
ax.set_title(f"{INSUMO}: últimos 90 días y pronóstico de 30")
guardar(fig, "05_pronostico_intervalo")
plt.show()

## 5. Qué aprendió el modelo (descomposición)

In [ ]:
comp = ex[ex["tipo"] == "history"]
paneles = [("tendencia", "Tendencia"), ("anual", "Estacionalidad anual"),
           ("semanal", "Estacionalidad semanal (últimas 4 semanas)"), ("feriados", "Efecto de feriados")]
fig, axes = plt.subplots(len(paneles), 1, figsize=(10, 9))
for ax, (col, titulo) in zip(axes, paneles):
    serie = comp.tail(28) if col == "semanal" else comp
    if serie[col].isna().all():
        ax.text(0.5, 0.5, "No detectado por el modelo", transform=ax.transAxes,
                ha="center", va="center", color=TEXTO_2)
    else:
        ax.plot(serie["fecha"], serie[col], color=PRONOSTICO, linewidth=1.5)
    ax.set_title(titulo, fontsize=11)
    eje_fechas(ax, con_anio=(col != "semanal"))
fig.suptitle(f"{INSUMO}: componentes detectados por ARIMA_PLUS", x=0.01, ha="left",
             fontsize=13, fontweight="bold", color=TEXTO)
fig.tight_layout(rect=(0, 0, 1, 0.97))
guardar(fig, "06_componentes")
plt.show()

## 6. Diagnóstico por insumo (`ML.ARIMA_EVALUATE`)

In [ ]:
d["arima"]